# TTC Station Reliability — ExplorationDiagnostic queries used to find the data quality problems that `pipeline.ipynb`corrects. Kept as a record of how each cleaning decision was reached.Run `pipeline.ipynb` first — every cell here reads tables it builds.

In [ ]:
import osimport globimport duckdbimport pandas as pdcon = duckdb.connect("ttc.duckdb")pd.set_option('display.max_rows', 100)

## File inventory

In [ ]:
paths = glob.glob("data/raw/**/*", recursive=True)files = [p for p in paths if os.path.isfile(p)]print(f"total files: {len(files)}")for f in sorted(files):    print(" ", f)

## GTFS route typesConfirms which mode code is the subway. 210 routes are buses, 20 arestreetcars/LRT, 3 are subway.

In [ ]:
con.sql("SELECT route_type, COUNT(*) AS routes FROM routes GROUP BY route_type ORDER BY routes DESC").df()

In [ ]:
con.sql("SELECT route_id, route_short_name, route_long_name FROM routes WHERE route_type = 1").df()

## parent_station is unpopulatedGTFS provides `parent_station` to group platforms into stations. TTC leaves itnull, so stations must be grouped on `stop_name` instead.

In [ ]:
con.sql("""    SELECT COUNT(*) AS total, COUNT(parent_station) AS has_parent    FROM stops    WHERE stop_id IN (SELECT DISTINCT stop_id FROM subway_stop_times)""").df()

## Denominator sanity checkInterchanges should be roughly double an ordinary station, and Line 4 stopsshould be lowest. Bloor-Yonge appearing at the bottom here was the symptomthat revealed GTFS splits it into two names.

In [ ]:
con.sql("SELECT * FROM station_trips_clean ORDER BY scheduled_trips DESC").df()

## Delay file schemasChecks for column drift across a decade of exports. All 10 files share thesame 10 columns; the csv adds `_id`.

In [ ]:
for f in sorted(glob.glob("data/raw/delay/*")):    if os.path.splitext(f)[1].lower() == '.xlsx':        d = pd.read_excel(f, nrows=0)    else:        d = pd.read_csv(f, nrows=0)    print(os.path.basename(f), d.columns.tolist())

## Station name concentrationHow many distinct station values cover what share of the rows. The top 70cover ~93%, the top 150 ~99% — so the remaining ~1,700 values are almost allsingletons, which is what makes exclusion rather than mapping defensible.

In [ ]:
con.sql("""WITH ranked AS (    SELECT        Station,        COUNT(*) AS n,        SUM(COUNT(*)) OVER (ORDER BY COUNT(*) DESC) AS running_total,        SUM(COUNT(*)) OVER (ORDER BY COUNT(*) DESC) / SUM(COUNT(*)) OVER () AS running_ratio,        ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC) AS rn    FROM raw_delays    GROUP BY Station)SELECT * FROM ranked WHERE rn IN (70, 150)""").df()

In [ ]:
con.sql("SELECT COUNT(DISTINCT Station) AS distinct_stations FROM raw_delays").df()

## Anti-join: delay names with no denominatorThe check that drives the crosswalk. Any station name here would silentlyvanish from the analysis, so each is either mapped or deliberately excluded.

In [ ]:
con.sql("""SELECT t1.Station, t1.Line, COUNT(*) AS nFROM raw_delays t1LEFT JOIN station_trips_clean t2 ON UPPER(t1.Station) = t2.stationWHERE t2.station IS NULLGROUP BY t1.Station, t1.LineORDER BY n DESCLIMIT 40""").df()

### Reverse: GTFS stations with no delays (should be empty)

In [ ]:
con.sql("""SELECT t2.stationFROM station_trips_clean t2LEFT JOIN clean_delays t1 ON t1.station = t2.stationWHERE t1.station IS NULL""").df()

## What got excluded~15,300 rows, 8%. Three categories: line-level records that name a line ratherthan a place, Line 3 stations with no current denominator, and facilities(yards, hostlers, carhouses, wyes, portals).

In [ ]:
con.sql("""SELECT t1.Station, t1.Line, COUNT(*) AS nFROM raw_delays t1LEFT JOIN clean_delays t2    ON t1.Station = t2.station AND t1.Line = t2.LineWHERE t2.station IS NULLGROUP BY t1.Station, t1.LineORDER BY n DESCLIMIT 30""").df()

### Segments are under 1% of rows, so excluding them cannot bias the ranking

In [ ]:
con.sql("""SELECT COUNT(*) AS segment_rows FROM raw_delays WHERE Station LIKE '% TO %'""").df()

## Zero-delay rows65% of raw rows have `Min Delay = 0` — logged incidents with no measurabledelay. Nearly all also have zero gap, meaning no service impact, which is whythey are excluded from the severity metrics.

In [ ]:
con.sql("""SELECT    COUNT(*) AS total,    SUM(CASE WHEN "Min Delay" = 0 THEN 1 ELSE 0 END) AS zero_delay,    SUM(CASE WHEN "Min Gap"   = 0 THEN 1 ELSE 0 END) AS zero_gap,    SUM(CASE WHEN "Min Delay" = 0 AND "Min Gap" > 0 THEN 1 ELSE 0 END) AS zero_delay_nonzero_gapFROM clean_delays""").df()

## Extreme delaysA handful of multi-hour records. The Jan 2026 and Feb 2025 clusters spanmultiple lines on consecutive days, so these are real system-wide disruptionsrather than entry errors. Retained; median and p95 are used instead of themean so they do not distort the severity ranking.

In [ ]:
con.sql("""SELECT station, Line, "Min Delay", "Min Gap", Code, TimestampFROM clean_delaysWHERE "Min Delay" > 400ORDER BY "Min Delay" DESC""").df()

## Sensitivity: full range vs 2018 onwardPositive `adjusted_diff` means the station ranked better in the full-rangeversion than it should have. The Vaughan extension stations dominate —Highway 407 by 21 places — confirming the full range understates them.

In [ ]:
con.sql("""SELECT    t1.station,    t1.raw_rank - t2.raw_rank AS raw_diff,    t1.adjusted_rank - t2.adjusted_rank AS adjusted_diffFROM station_unreliability t1JOIN station_unreliability_2018 t2 ON t1.station = t2.stationORDER BY adjusted_diff DESCLIMIT 20""").df()

## Survivor countOf the 10 worst stations by raw delay count, how many remain in the top 10after adjusting for scheduled traffic.

In [ ]:
con.sql("""SELECT COUNT(*) AS survivorsFROM station_unreliability_2018WHERE raw_rank <= 10 AND adjusted_rank <= 10""").df()

In [ ]:
con.sql("""SELECT station, raw_rank, adjusted_rank, deltaFROM station_unreliability_2018WHERE raw_rank <= 10 OR adjusted_rank <= 10ORDER BY adjusted_rank""").df()

## Frequency vs severityp50 is flat at 4–5 minutes network-wide: a typical delay is the same lengtheverywhere. p95 ranges 12 to 27 and runs opposite to the delay rate — theterminals have frequent mild delays, quieter stations have rare severe ones.

In [ ]:
con.sql("""SELECT station, delay_rate, p50, p95, avg_delay, adjusted_rankFROM station_unreliability_2018ORDER BY p95 DESC""").df()

## GTFS feed window`calendar.txt` gives the period the snapshot covers — late July to earlySeptember 2026, about six weeks of summer service. This is the basis for thecaveat that the denominator is a scaled snapshot, not historical service.

In [ ]:
con.sql("CREATE OR REPLACE TABLE calendar AS SELECT * FROM read_csv_auto('data/raw/schedules/calendar.txt')")con.sql("SELECT * FROM calendar").df()